# Model Training — Sri Lanka Property Price Predictor

This notebook trains a **CatBoost Regressor** to predict property prices.

**Why CatBoost?**
- Handles categorical features natively (City, District)
- Robust to overfitting with ordered boosting
- No need for extensive feature scaling
- Excellent performance on tabular data with mixed feature types

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostRegressor

DATA_DIR = os.path.join('..', 'data')
MODEL_DIR = os.path.join('..', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

## 1. Load Cleaned Data

In [2]:
df = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_data.csv'))
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (15003, 7)


,Bedrooms,Bathrooms,Land_Size_Perches,House_Size_Sqft,City,District,Price_LKR
0,3,1,50.0,1600.0,109,15,5400000.0
1,3,3,8.0,1480.0,10,4,16800000.0
2,3,2,20.0,2800.0,92,6,20000000.0
3,5,5,22.0,4000.0,36,4,187000000.0
4,4,4,11.0,3300.0,152,4,55000000.0


## 2. Encode Categorical Features

In [3]:
feature_columns = ['Bedrooms', 'Bathrooms', 'Land_Size_Perches', 'House_Size_Sqft', 'City', 'District']
target_column = 'Price_LKR'
categorical_cols = ['City', 'District']

# Label encode categorical features
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f'Encoded {col}: {len(le.classes_)} unique values')

df[feature_columns].head()

Encoded City: 170 unique values
Encoded District: 24 unique values


,Bedrooms,Bathrooms,Land_Size_Perches,House_Size_Sqft,City,District
0,3,1,50.0,1600.0,12,7
1,3,3,8.0,1480.0,2,18
2,3,2,20.0,2800.0,162,20
3,5,5,22.0,4000.0,100,18
4,4,4,11.0,3300.0,60,18


## 3. Train-Test Split

In [4]:
X = df[feature_columns]
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'\nFeatures: {feature_columns}')
print(f'Target: {target_column}')

Training set: 12002 samples
Test set:     3001 samples

Features: ['Bedrooms', 'Bathrooms', 'Land_Size_Perches', 'House_Size_Sqft', 'City', 'District']
Target: Price_LKR


## 4. Train CatBoost Regressor

### Hyperparameters
| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `iterations` | 1000 | Sufficient for convergence with early stopping |
| `learning_rate` | 0.05 | Moderate rate for stable learning |
| `depth` | 8 | Deep enough to capture feature interactions |
| `l2_leaf_reg` | 3 | Regularization to prevent overfitting |
| `early_stopping_rounds` | 50 | Stop if no improvement for 50 rounds |

In [5]:
cat_features_indices = [feature_columns.index(c) for c in categorical_cols]

model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=100,
    eval_metric='RMSE',
    early_stopping_rounds=50,
    cat_features=cat_features_indices,
)

model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    verbose=100
)

print('\nTraining complete!')

0:	learn: 27409242.0770860	test: 28638621.7539287	best: 28638621.7539287 (0)	total: 233ms	remaining: 3m 52s
100:	learn: 12277503.6995603	test: 13729638.7530377	best: 13729638.7530377 (100)	total: 8.95s	remaining: 1m 19s
200:	learn: 11381966.5460237	test: 13164856.2518809	best: 13164856.2518809 (200)	total: 16.5s	remaining: 1m 5s
300:	learn: 10731610.1541490	test: 12827730.5094691	best: 12827730.5094691 (300)	total: 22.1s	remaining: 51.4s
400:	learn: 10230271.8279192	test: 12599979.5821075	best: 12599979.5821075 (400)	total: 27.7s	remaining: 41.4s
500:	learn: 9830427.4358392	test: 12396048.5769918	best: 12393995.3772765 (499)	total: 33.8s	remaining: 33.6s
600:	learn: 9410651.5530144	test: 12180587.0823870	best: 12180587.0823870 (600)	total: 39.5s	remaining: 26.2s
700:	learn: 9115380.8769726	test: 12038727.4612027	best: 12038727.4612027 (700)	total: 45.2s	remaining: 19.3s
800:	learn: 8844831.4569892	test: 11924839.3519833	best: 11924839.3519833 (800)	total: 50.7s	remaining: 12.6s
900:	le

## 5. Save Model & Preprocessor as .pkl

In [6]:
# Save model
MODEL_PATH = os.path.join(MODEL_DIR, 'catboost_model.pkl')
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(model, f)
print(f'Model saved to: {MODEL_PATH}')

# Save preprocessor (label encoders + config)
PREPROCESSOR_PATH = os.path.join(MODEL_DIR, 'preprocessor.pkl')
preprocessor = {
    'label_encoders': label_encoders,
    'feature_columns': feature_columns,
    'categorical_cols': categorical_cols,
    'target_column': target_column,
}
with open(PREPROCESSOR_PATH, 'wb') as f:
    pickle.dump(preprocessor, f)
print(f'Preprocessor saved to: {PREPROCESSOR_PATH}')

Model saved to: ..\models\catboost_model.pkl
Preprocessor saved to: ..\models\preprocessor.pkl


In [7]:
# Save test data for evaluation notebook
test_data = pd.DataFrame(X_test, columns=feature_columns)
test_data['Price_LKR'] = y_test.values
test_data.to_csv(os.path.join(DATA_DIR, 'test_data.csv'), index=False)

train_data = pd.DataFrame(X_train, columns=feature_columns)
train_data['Price_LKR'] = y_train.values
train_data.to_csv(os.path.join(DATA_DIR, 'train_data.csv'), index=False)

print('Train and test data saved for evaluation.')

Train and test data saved for evaluation.


## 6. Why CatBoost Over Other Models?

| Model | Pros | Cons for this problem |
|-------|------|-----------------------|
| **Linear Regression** | Simple, interpretable | Cannot capture non-linear price patterns (e.g., location premiums) |
| **Random Forest** | Good for mixed features | Slower, requires manual categorical encoding |
| **XGBoost** | Excellent performance | Requires manual categorical encoding |
| **CatBoost** ✅ | Native categorical support, ordered boosting, robust | Slightly slower training |

CatBoost is chosen because:
1. **Native categorical handling** — City and District don't need one-hot encoding
2. **Ordered boosting** — reduces overfitting compared to classical boosting
3. **Strong performance** on medium-sized tabular data
4. **Built-in feature importance** and SHAP compatibility for XAI